In [ ]:
import math
import random
import sympy

コードブックの作成:Unicodeのひらがなブロック、英数記号ブロック、カタカナブロック、ギリシャ文字ブロック、キリル文字ブロックからコードブック辞書の作成

＊＊各ブロックの最小最大をタプルに格納して、ループを回す方がシンプル

***暗号文が文字で表現される前提で作っているので、コードブックで使用するキャラクタは画面表示可能なもののみ。

In [1]:
def generate_codebook():
  hiragana_base=0x3041
  codebook={}
  invcbook={}
  i=0
  #ひらがな領域
  while i+hiragana_base<=0x3096:
    codebook[chr(i+hiragana_base)]=i
    invcbook[i]=chr(i+hiragana_base)
    i+=1
  #print('number of hiragana code + space:A.K.A. minimum:N-1',i)
  minN=i
  j=0
  #記号・数字・アルファベット
  while j+0x0020<=0x007e:
    codebook[chr(j+0x0020)]=i
    invcbook[i]=chr(j+0x0020)
    i+=1
    j+=1
  bsetN=i
  j=0
  #カタカナ領域
  while j+0x30a1<=0x30fa:
    codebook[chr(j+0x30a1)]=i
    invcbook[i]=chr(j+0x30a1)
    i+=1
    j+=1
  b2setN=StopAsyncIteration
  j=0
  #greek
  while j+0x03b1<=0x03c9:
    codebook[chr(j+0x03b1)]=i
    invcbook[i]=chr(j+0x03b1)
    i+=1
    j+=1
  j=0
#cyrillic
  while j+0x0410<=0x044f:
    codebook[chr(j+0x0410)]=i
    invcbook[i]=chr(j+0x0410)
    i+=1
    j+=1
  maxN=i
  return codebook,invcbook,minN,maxN

RSA暗号学習用Pythonコード
このノートはRSA暗号の仕組みを学習するためのステップbyステップ実装です。
文字by文字で暗号を解読する仕様です。

A÷Bのあまり（余剰）計算をmod(A,B)と記述する。

AとBの間で暗号通信

Aの公開鍵 $N_A,E_A$ 秘密鍵を$D_A$ Bの公開鍵$N_B,E_B$ 秘密鍵$D_B$ とする。

AからBの平文$M_{AB}$ BからAへの平文　$M_{BA}$

暗号文はそれぞれ$C_{AB}, C_{BA}$


１．A->B　暗号化　$C_{AB}=mod({M_{AB}}^{E_B},{N_B})$: AはBの公開鍵で暗号作成

２．$M_{AB}=mod({C_{AB}}^{D_B},N_{B})$ Bは自分の秘密鍵で復号化

指数関数は　math ライブラリのpow関数で　$X^y$: pow(X,y) もしくは　X**y

余剰計算xをyで割ったときの余り　mod(x,y):x%y 商は x//y 通常の割り算はx/y

ちなみに、pow関数で第３因数zを追加すると、指数計算した後にｚのモッドを計算する

$mod({C}^{D},N)$:mod(C,D,N)


In [3]:
#暗号化関数
#pubric key は送信先からもらう。message:C public_key N,E
def encrypt(messagechar, public_key):
    ne, ee = public_key
    messagechar = int(messagechar)
    ne=int(ne)
    ee=int(ee)
    return pow(messagechar, ee, ne)

#復号化関数
#ciphertext:C private_key N,D
def decrypt(cipherc, ds,ns):
    cipherc=int(cipherc)
    ds=int(ds)
    ns=int(ns)
    return pow(cipherc, ds, ns)

N,E,Dの作成
二つの素数の選択P,Q

1.key1_o N=PQ

Nで割った余りがテキストコードなので…

コードブックのひらがなとスペース部分の数をNminとすると N-1>=Nmin

暗号もテキストで表現するので、不必要な番号部分にも文字情報が必要なので、コードブックの最大値Nmax とすると、N＜Nmax

2. P-1とQ-1の最小公倍数の計算　L＝LCM(P-1,Q-1)
3.EとDの積を任意のnを使って　ED=nL+1　と定義する。

この時、ＥＤとＬが互いに素である必要があるので最大公倍数gcd(ED,L)==1である必要がある。

４．　ＥＤを素因数分解して異なる二つの積となるものを選ぶ。$x^ay^b $ a,b>1若しくは$x^a$ a>2のものを選択する必要がある。


In [4]:
#maxNまでの素数
def get_primes_up_to(maxN):
    primes = list(sympy.primerange(3,maxN))
    return primes
#コードブックの範囲でとりえるP,Qリスト
def find_pq_pairs(primes,minN,maxN):
    result = []
    for i in range(len(primes)):
        for j in range(i + 1, len(primes)):
            p = primes[i]
            q = primes[j]
            pq_minus_1 = p * q - 1
            if minN < pq_minus_1 < maxN:
                result.append((p, q,p*q))
    return result

def find_keypaircandidate(P,Q,maxN):
    cnddt=[]
    L=math.lcm(P-1,Q-1)
    for n in range(1,maxN):
      ED=n*L+1
      factors=sympy.factorint(ED)
      fact2=list(factors.items())
      if math.gcd(ED,L)==1:
        if len(factors)>2:
          cnddt.append((ED,fact2))
        elif any(fact>2 for fact in factors.values()):
          cnddt.append((ED,fact2))
    return cnddt



コードブックの生成

In [5]:
codebook,invcbook,minN,maxN=generate_codebook()
print('number of hiragana code + space:A.K.A. minimum:N-1',minN)
print('maximum number for this code book',maxN)
if input("do you wanna see your codebook:if yes type y")=='y':
  print(codebook)

number of hiragana code + space:A.K.A. minimum:N-1 86
maximum number for this code book 360
do you wanna see your codebook:if yes type yy
{'ぁ': 0, 'あ': 1, 'ぃ': 2, 'い': 3, 'ぅ': 4, 'う': 5, 'ぇ': 6, 'え': 7, 'ぉ': 8, 'お': 9, 'か': 10, 'が': 11, 'き': 12, 'ぎ': 13, 'く': 14, 'ぐ': 15, 'け': 16, 'げ': 17, 'こ': 18, 'ご': 19, 'さ': 20, 'ざ': 21, 'し': 22, 'じ': 23, 'す': 24, 'ず': 25, 'せ': 26, 'ぜ': 27, 'そ': 28, 'ぞ': 29, 'た': 30, 'だ': 31, 'ち': 32, 'ぢ': 33, 'っ': 34, 'つ': 35, 'づ': 36, 'て': 37, 'で': 38, 'と': 39, 'ど': 40, 'な': 41, 'に': 42, 'ぬ': 43, 'ね': 44, 'の': 45, 'は': 46, 'ば': 47, 'ぱ': 48, 'ひ': 49, 'び': 50, 'ぴ': 51, 'ふ': 52, 'ぶ': 53, 'ぷ': 54, 'へ': 55, 'べ': 56, 'ぺ': 57, 'ほ': 58, 'ぼ': 59, 'ぽ': 60, 'ま': 61, 'み': 62, 'む': 63, 'め': 64, 'も': 65, 'ゃ': 66, 'や': 67, 'ゅ': 68, 'ゆ': 69, 'ょ': 70, 'よ': 71, 'ら': 72, 'り': 73, 'る': 74, 'れ': 75, 'ろ': 76, 'ゎ': 77, 'わ': 78, 'ゐ': 79, 'ゑ': 80, 'を': 81, 'ん': 82, 'ゔ': 83, 'ゕ': 84, 'ゖ': 85, ' ': 86, '!': 87, '"': 88, '#': 89, '$': 90, '%': 91, '&': 92, "'": 93, '(': 94, ')': 95, '*': 96

単純替え字暗号 暗号化

In [6]:
text=input("暗号化したい文章を打ってください。")
scrypt=[]
for i in range(len(text)):
    scrypt.append(codebook[text[i]])
print(scrypt)

暗号化したい文章を打ってください。おはよう
[9, 46, 71, 5]


単純替え字、復号化

In [7]:
scode=list(map(int, input("数値をカンマ区切りで複数入力して下さい ").split(",")))
for i in range(len(scode)):
  print(invcbook[scode[i]],end='')


数値をカンマ区切りで複数入力して下さい 9, 46, 71, 5
おはよう

シーザー暗号化

In [8]:
ssft=input("シーザーシフトを入れてください。")
shift=int(ssft)
for i in range(len(scrypt)):
  print(invcbook[(scrypt[i]+shift)%maxN],end='')



シーザーシフトを入れてください。30
とろ/つ

シーザー復号化

In [10]:
issft=input("複合に使うシーザーシフトを入れてください:")
ishift=int(issft)
sctxt=input("シーザー暗号を入れてください:")
for i in range(len(sctxt)):
  c=codebook[sctxt[i]]
  c=(c-ishift+maxN)%maxN
  print(invcbook[c],end='')

複合に使うシーザーシフトを入れてください:30 
シーザー暗号を入れてください:とろ/つ
おはよう

ビジュネル暗号：平文のi文字目のシーザーシフトをキーから得られる数字で行う。
たとえば、”りてらしいはいけめんせんせい”が平文　キーが”おはよう”の場合　おはようの文字数は4　”お”のシフトは9、”は”は46、”よ”は71、”う”は５

一文字目の”り”はシフト９　”て”は46　”ら”は71　”し”は　５のシフトを行う。

キーの長さ（今は４）を超えた場合は繰り返し
”い”＋9　”は”＋４６　”い”＋71　”け”＋５

In [ ]:
vkey=input("vigener暗号のキーを入れてください。:")
lkey=len(vkey)
svkey=[]
for i in range(lkey):
  svkey.append(codebook[vkey[i]])
for i in range(len(scrypt)):
  print(invcbook[(scrypt[i]+svkey[i%lkey])%maxN],end='')

vigener暗号のキーを入れてください。:いけめん
きゔ"Z

ビジュネル復号化

In [ ]:
vkey2=input("復号用のvigener暗号のキーを入れてください。:")
cvtxt=input("復号したいビジュネル暗号を入れてください。:")
lvkey=len(vkey2)
svkey2=[]
for i in range(lvkey):
  svkey2.append(codebook[vkey2[i]])
for i in range(len(cvtxt)):
  print(invcbook[(codebook[cvtxt[i]]-svkey2[i%lvkey]+maxN)%maxN],end='')

復号用のvigener暗号のキーを入れてください。kome
復号したいビジュネル暗号を入れてください。tタブj
おはよう

RSA暗号キー作成
＊＊PQの候補の選択を出席番号で抽出

In [ ]:
print("コードブックの最大値より小さい素数のリストを示しますか？")
primes=get_primes_up_to(maxN)
print(primes)
pq_pair=find_pq_pairs(primes,minN,maxN)
print("このコードブックで使えるPAペアのリスト P,Q,N")
print((pq_pair))
idnum=input("あなたの出席番号は？")
idnum=int(idnum)
numkey=idnum%len(pq_pair)#出席番号から何番目の候補を使うか決定
print("your P Q  and N is",pq_pair[numkey])
P,Q,N=pq_pair[numkey]#候補の中からPQNを決定
#print(P,Q,N)
#P=
#Q=  この2行でP,Qを候補から選んで直接入力でも良い。
#N=P*Q
candidate=find_keypaircandidate(P,Q,30)#選択できるチョイスが無い場合は30を大きくするともしかして。…
print("result (ED,[(fact,power)...(fact,power)])....")
print(candidate)
E_y=input("yourchoice E:")
D_y=input("yourchoice D:")
print("public key of yours:tell these to friends",N,E_y)
print("secret key of yours",D_y)

コードブックの最大値より小さい素数のリストを示しますか？
[3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311, 313, 317, 331, 337, 347, 349, 353, 359]
このコードブックで使えるPAペアのリスト P,Q,N
[(3, 31, 93), (3, 37, 111), (3, 41, 123), (3, 43, 129), (3, 47, 141), (3, 53, 159), (3, 59, 177), (3, 61, 183), (3, 67, 201), (3, 71, 213), (3, 73, 219), (3, 79, 237), (3, 83, 249), (3, 89, 267), (3, 97, 291), (3, 101, 303), (3, 103, 309), (3, 107, 321), (3, 109, 327), (3, 113, 339), (5, 19, 95), (5, 23, 115), (5, 29, 145), (5, 31, 155), (5, 37, 185), (5, 41, 205), (5, 43, 215), (5, 47, 235), (5, 53, 265), (5, 59, 295), (5, 61, 305), (5, 67, 335), (5, 71, 355), (7, 13, 91), (7, 17, 119), (7, 19, 133), (7, 23, 161), (7, 29, 203), (7, 31, 217), (7, 37, 259), (7, 41, 287), (7, 43, 301), (7, 47, 329), (11, 13, 1

他人のパブリックキーでのRSA暗号化

In [ ]:
Np=input("暗号化に使う公開鍵１（N)：")
Ep=input("暗号化に使う公開鍵２（E)：")
pubkey=(Np,Ep)
for i in range(len(scrypt)):
  c=encrypt(scrypt[i],pubkey)
  #print(c,end='')
  print(invcbook[encrypt(scrypt[i], pubkey)],end='')

暗号化に使う公開鍵１（N)：141
暗号化に使う公開鍵２（E)：77
れRぅぎ

RSA復号化　秘密鍵使用

In [ ]:
cyper=input("暗号を入れてください：")
for i in range(len(cyper)):
  print(invcbook[decrypt(codebook[cyper[i]],D_y,N)],end='')

暗号を入れてください：お.よみ
おはよう